# Wavelet Transform Based Restoration of Damaged Chinese Painting
**Digital Image Processing — Case Study (Google Colab notebook)**

**Aim:** To study and implement the Wavelet Transform for digital image processing, with the objective of reducing noise and restoring degraded images by decomposing an image into different frequency components, and to evaluate the reconstructed image using MSE, PSNR and SSIM.

### How to run
1. `Runtime → Run all` (or press **Ctrl + F9**). No GPU needed — the default CPU runtime is enough.
2. By default the notebook downloads a public-domain Chinese painting (Ma Yuan, *Walking on a Mountain Path in Spring*, Southern Song dynasty, Wikimedia Commons). To use your own painting, set `IMAGE_SOURCE = "upload"` in the **Configuration** cell.
3. Every figure and table is saved to `/content/results/`, and the last cell zips it all and downloads `dip_wavelet_results.zip` — use these directly in the report and PPT.

### Pipeline
```
Original painting → simulated tears / scratches / noise → damaged digital image
→ 3.1 Pre-processing → 3.2 Damage detection → 3.3 Segmentation
→ 3.4 DWT (LL, LH, HL, HH) → 3.5 Wavelet thresholding (BayesShrink denoising)
→ 3.6 Inverse DWT → 3.7 Missing-region reconstruction (inpainting) → 3.8 Post-processing
→ Restored painting → Evaluation (MSE, PSNR, SSIM)
```
*Why damage is simulated:* MSE/PSNR/SSIM need a clean reference. Starting from an undamaged painting and damaging it in a controlled, reproducible way (fixed random seed) gives us that ground truth. The optional last section restores an already-damaged photo where no reference exists.

## 0. Setup

In [ ]:
!pip -q install PyWavelets scikit-image opencv-python-headless

import os, sys, io, time, zipfile, warnings
import numpy as np
import cv2, pywt, requests
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
OUT_DIR = "/content/results" if IN_COLAB else "results"
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": False, "grid.color": "#e4e3df", "grid.linewidth": 0.8,
})
BLUE, ORANGE, AQUA, GREY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#b9b7b0", "#0b0b0b"

print("Running in Colab:", IN_COLAB)
print("NumPy", np.__version__, "| OpenCV", cv2.__version__, "| PyWavelets", pywt.__version__)
print("Results folder:", OUT_DIR)

## 1. Configuration
All tunable parameters live here. Change a value, then `Runtime → Run all` again.

In [ ]:
# ---------- input image ----------
IMAGE_SOURCE = "sample"   # "sample" (public-domain painting) | "upload" (your clean image) | "path"
IMAGE_PATH   = ""         # used only when IMAGE_SOURCE == "path", e.g. "/content/my_painting.jpg"
MAX_SIDE     = 768        # longer side after resizing (keeps everything fast)
SEED         = 42         # fixed seed -> the same damage pattern every run (reproducible results)

# ---------- simulated damage ----------
HOLE_COVERAGE = 0.10      # fraction of the painting lost to torn / missing patches
N_TEARS       = 4         # long irregular tears / cracks
N_SCRATCHES   = 6         # thin straight scratches
NOISE_SIGMA   = 0.05      # Gaussian scan noise (std on a 0-1 intensity scale)
FADING        = 0.0       # 0 = off; e.g. 0.2 adds global yellowing / contrast loss

# ---------- 3.1-3.3 detection and segmentation ----------
V_THR, S_THR  = 225, 40   # missing regions: bright (HSV value >= V_THR) and unsaturated (S <= S_THR)
TOPHAT_THR    = 28        # thin bright scratches: white top-hat response threshold
MIN_AREA      = 30        # ignore detected specks smaller than this (pixels)
MASK_MARGIN   = 2         # safety margin (pixels) added around every damaged region

# ---------- 3.4-3.6 wavelet ----------
WAVELET       = "db4"     # try: haar, db2, db4, sym4, coif2, bior2.2
LEVEL         = 3         # decomposition levels
THRESH_MODE   = "soft"    # soft | hard
THRESH_RULE   = "bayes"   # bayes (BayesShrink) | visu (VisuShrink / universal threshold)
DETAIL_GAIN   = 1.0       # >1 boosts surviving detail coefficients (detail enhancement)

# ---------- 3.7-3.8 reconstruction and post-processing ----------
INPAINT_RADIUS = 5
CLAHE_CLIP     = 1.0      # display enhancement only (reported separately)
UNSHARP_AMOUNT = 0.3

## 2. Helper functions (I/O, metrics, plotting)

In [ ]:
def to_f(img):  return img.astype(np.float32) / 255.0
def to_u8(img): return np.clip(img * 255.0 + 0.5, 0, 255).astype(np.uint8)

def save_fig(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.show()
    print("saved ->", path)

def show_row(images, titles, name, cmap=None, height=4.2):
    n = len(images)
    h, w = images[0].shape[:2]
    fig, axes = plt.subplots(1, n, figsize=(height * w / h * n, height + 0.5))
    axes = np.atleast_1d(axes)
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im, cmap=cmap if im.ndim == 2 else None, vmin=0, vmax=1 if im.dtype != np.uint8 else 255)
        ax.set_title(t); ax.axis("off")
    plt.tight_layout()
    save_fig(fig, name)

def evaluate(ref, img, region=None):
    # MSE on the 0-255 scale, PSNR in dB, SSIM in [0, 1]. region = optional boolean mask
    a = to_u8(ref).astype(np.float64); b = to_u8(img).astype(np.float64)
    if region is not None:
        mse = np.mean((a[region] - b[region]) ** 2)
        return 10 * np.log10(255 ** 2 / max(mse, 1e-10))
    mse  = np.mean((a - b) ** 2)
    psnr = 10 * np.log10(255 ** 2 / max(mse, 1e-10))
    ssim = structural_similarity(a, b, channel_axis=2, data_range=255)
    return mse, psnr, ssim

def prepare(img, max_side=MAX_SIDE, multiple=16):
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    img = cv2.resize(img, (int(round(w * s)), int(round(h * s))),
                     interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    h, w = img.shape[:2]
    return img[: h - h % multiple, : w - w % multiple].copy()   # size divisible by 2**LEVEL

def densest_window(mask, size=160):
    # top-left corner of the window that contains the most damage (for zoomed crops)
    size = min(size, mask.shape[0], mask.shape[1])
    s = cv2.boxFilter(mask.astype(np.float32), -1, (size, size), normalize=False, borderType=cv2.BORDER_CONSTANT)
    half = size // 2
    s[:half, :] = s[-half:, :] = 0; s[:, :half] = s[:, -half:] = 0
    y, x = np.unravel_index(np.argmax(s), s.shape)
    return max(y - half, 0), max(x - half, 0), size

## 3. Load the original (undamaged) painting

In [ ]:
SAMPLE_URLS = [
    "https://upload.wikimedia.org/wikipedia/commons/c/c2/Ma_Yuan_Walking_on_Path_in_Spring.jpg",
    "https://commons.wikimedia.org/wiki/Special:FilePath/Ma_Yuan_Walking_on_Path_in_Spring.jpg",
]
HEADERS = {"User-Agent": "DIP-CaseStudy-Colab/1.0 (student project; educational use)"}

def decode(buf):
    img = cv2.imdecode(np.frombuffer(buf, np.uint8), cv2.IMREAD_COLOR)
    if img is None: raise ValueError("could not decode image")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def load_image():
    if IMAGE_SOURCE == "path":
        return cv2.cvtColor(cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB), os.path.basename(IMAGE_PATH)
    if IMAGE_SOURCE == "upload" and IN_COLAB:
        from google.colab import files
        up = files.upload()
        name = next(iter(up))
        return decode(up[name]), name
    for url in SAMPLE_URLS:
        try:
            r = requests.get(url, headers=HEADERS, timeout=30); r.raise_for_status()
            return decode(r.content), "Ma Yuan - Walking on a Mountain Path in Spring (public domain)"
        except Exception as e:
            print("download failed:", url, "->", e)
    if IN_COLAB:
        print("Sample download failed - please upload a clean painting image instead.")
        from google.colab import files
        up = files.upload(); name = next(iter(up))
        return decode(up[name]), name
    from skimage import data
    print("Falling back to a scikit-image test picture.")
    return data.coffee(), "scikit-image coffee (fallback)"

raw, IMAGE_NAME = load_image()
original = to_f(prepare(raw))
H, W = original.shape[:2]
cv2.imwrite(os.path.join(OUT_DIR, "00_original.png"), cv2.cvtColor(to_u8(original), cv2.COLOR_RGB2BGR))
print(f"Image: {IMAGE_NAME}\nOriginal size: {raw.shape[1]}x{raw.shape[0]}  ->  working size: {W}x{H}")
plt.figure(figsize=(7, 7 * H / W)); plt.imshow(original); plt.axis("off"); plt.title("Original (undamaged) painting"); plt.show()

## 4. Simulate physical damage → *damaged digital image*
Models what happens to an old scroll: **torn / missing patches** (exposed backing paper), **long cracks / tears**, **thin scratches**, **scan noise** and optional **fading**. The exact damage mask is kept as ground truth for evaluating detection.

In [ ]:
rng = np.random.default_rng(SEED)

def blob_mask(h, w, coverage, scale=10):
    coarse = rng.random((max(h // scale, 4), max(w // scale, 4))).astype(np.float32)
    field = cv2.GaussianBlur(cv2.resize(coarse, (w, h), interpolation=cv2.INTER_CUBIC), (0, 0), 18)
    field = (field - field.min()) / (np.ptp(field) + 1e-8)
    fine = cv2.GaussianBlur(rng.random((h, w)).astype(np.float32), (0, 0), 1.5)
    field = field + 0.06 * (fine - 0.5)                       # jagged, paper-like torn edges
    return field > np.quantile(field, 1 - coverage)

def random_walks(h, w, n, thickness, step, length, straight=False):
    m = np.zeros((h, w), np.uint8)
    for _ in range(n):
        x, y = rng.uniform(0, w), rng.uniform(0, h); ang = rng.uniform(0, 2 * np.pi); pts = [(x, y)]
        for _ in range(length):
            ang += rng.normal(0, 0.05 if straight else 0.35)
            x += step * np.cos(ang); y += step * np.sin(ang); pts.append((x, y))
        cv2.polylines(m, [np.int32(pts)], False, 1, thickness, cv2.LINE_8)
    return m.astype(bool)

def simulate_damage(img, coverage=HOLE_COVERAGE, noise_sigma=NOISE_SIGMA, fading=FADING):
    h, w = img.shape[:2]
    holes   = blob_mask(h, w, coverage)
    tears   = random_walks(h, w, N_TEARS, 4, 6, 45)
    scratch = random_walks(h, w, N_SCRATCHES, 2, 8, 30, straight=True)
    gt = holes | tears | scratch
    dmg = img.copy()
    if fading > 0:
        dmg = (1 - fading) * dmg + fading * np.array([0.80, 0.72, 0.55], np.float32)
    dmg[gt] = np.array([0.965, 0.955, 0.93], np.float32)     # exposed backing paper
    dmg = dmg + rng.normal(0, noise_sigma, dmg.shape).astype(np.float32)
    return np.clip(dmg, 0, 1).astype(np.float32), gt

damaged, gt_mask = simulate_damage(original)
cv2.imwrite(os.path.join(OUT_DIR, "00_damaged.png"), cv2.cvtColor(to_u8(damaged), cv2.COLOR_RGB2BGR))
print(f"Damaged area (ground truth): {gt_mask.mean() * 100:.2f}% of the painting")
show_row([original, damaged], ["1. Original painting", "2. Damaged painting (input)"], "fig01_original_vs_damaged.png")

## 3.1 Pre-processing
Grayscale conversion and noise reduction. These smoothed copies are used **only for detection**; the restoration itself works on the full-colour damaged image.

In [ ]:
def preprocess(img):
    u8 = to_u8(img)
    gray     = cv2.cvtColor(u8, cv2.COLOR_RGB2GRAY)
    smooth   = cv2.GaussianBlur(u8, (0, 0), 1.2)    # suppresses noise for region detection
    median   = cv2.medianBlur(u8, 3)                # edge-preserving, keeps thin scratches
    return gray, smooth, median

gray, smooth, median = preprocess(damaged)
show_row([gray, cv2.cvtColor(smooth, cv2.COLOR_RGB2GRAY)], ["3.1a Grayscale", "3.1b Noise-reduced (Gaussian)"],
         "fig02_preprocessing.png", cmap="gray")

## 3.2 Damage detection
* **Missing / torn regions** → pixels that are very bright **and** unsaturated in HSV (exposed paper), after smoothing.
* **Thin scratches and cracks** → morphological **white top-hat** (bright structures thinner than the 9×9 element), restricted to bright, low-saturation pixels.

In [ ]:
def keep_large(mask, min_area):
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    keep = np.zeros(n, bool); keep[1:] = st[1:, cv2.CC_STAT_AREA] >= min_area
    return keep[lab]

def detect_damage(smooth, median):
    hsv = cv2.cvtColor(smooth, cv2.COLOR_RGB2HSV)
    regions = (hsv[..., 2] >= V_THR) & (hsv[..., 1] <= S_THR)
    g = cv2.cvtColor(median, cv2.COLOR_RGB2GRAY)
    tophat = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    hsv_m = cv2.cvtColor(median, cv2.COLOR_RGB2HSV)
    lines = (tophat > TOPHAT_THR) & (hsv_m[..., 2] > 200) & (hsv_m[..., 1] < 70)
    lines = cv2.morphologyEx(lines.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    lines = keep_large(lines, 15)
    return regions, lines, tophat

regions, lines, tophat = detect_damage(smooth, median)
raw_mask = regions | lines
show_row([regions.astype(np.float32), tophat / max(tophat.max(), 1), raw_mask.astype(np.float32)],
         ["3.2a Bright/unsaturated regions", "3.2b White top-hat response", "3.2c Raw damage map (white = damaged)"],
         "fig03_damage_detection.png", cmap="gray")

## 3.3 Segmentation — separate damaged regions from valid regions
Morphological closing, removal of tiny specks, connected-component labelling, and a small safety margin around each region. Detection quality is measured against the ground-truth mask, and a **false-alarm check** runs the same detector on the *undamaged* original.

In [ ]:
def segment(raw):
    m = cv2.morphologyEx(raw.astype(np.uint8), cv2.MORPH_CLOSE,
                         cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    m = keep_large(m, MIN_AREA).astype(np.uint8)
    if MASK_MARGIN > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * MASK_MARGIN + 1,) * 2)
        m = cv2.dilate(m, k)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    return m.astype(bool), n - 1, stats[1:, cv2.CC_STAT_AREA], lab

mask, n_regions, region_areas, labels = segment(raw_mask)

# detection quality vs ground truth
tp = (mask & gt_mask).sum(); fp = (mask & ~gt_mask).sum(); fn = (~mask & gt_mask).sum()
precision, recall = tp / (tp + fp), tp / (tp + fn)
iou, dice = tp / (tp + fp + fn), 2 * tp / (2 * tp + fp + fn)
_, s0, m0 = preprocess(original)
r0, l0, _ = detect_damage(s0, m0)
false_alarm = segment(r0 | l0)[0].mean()

det_df = pd.DataFrame({
    "Metric": ["Damaged area - ground truth (%)", "Damaged area - detected (%)", "Damaged regions found",
               "Precision", "Recall", "IoU", "Dice", "False alarm on undamaged original (%)"],
    "Value": [gt_mask.mean() * 100, mask.mean() * 100, n_regions, precision, recall, iou, dice, false_alarm * 100]})
det_df["Value"] = det_df["Value"].round(3)
det_df.to_csv(os.path.join(OUT_DIR, "table_detection_metrics.csv"), index=False)
display(det_df)

overlay = damaged.copy(); overlay[mask] = 0.6 * overlay[mask] + 0.4 * np.array([0.89, 0.12, 0.12])
errmap = np.zeros((H, W, 3), np.float32)
errmap[mask & gt_mask] = [1, 1, 1]; errmap[mask & ~gt_mask] = [0.92, 0.41, 0.20]; errmap[~mask & gt_mask] = [0.16, 0.47, 0.84]
show_row([mask.astype(np.float32), overlay, errmap],
         ["3.3a Final mask (white = damaged)", f"3.3b Segmentation: red = damaged ({n_regions} regions)",
          "3.3c vs truth: white=TP, orange=FP, blue=FN"], "fig04_segmentation.png", cmap="gray")

## 3.4 Wavelet decomposition (DWT)
The 2-D DWT splits the image into an **approximation** sub-band (LL, low frequencies – overall shapes and colour washes) and three **detail** sub-bands (LH horizontal, HL vertical, HH diagonal – edges, brush strokes **and noise**). Repeating on LL gives a multi-level pyramid.

In [ ]:
gray_f = to_f(gray)
LL, (LH, HL, HH) = pywt.dwt2(gray_f, WAVELET, mode="periodization")

def norm_vis(c, detail=True):
    if detail:
        c = np.abs(c); return np.clip(c / (np.percentile(c, 99.5) + 1e-8), 0, 1)
    return (c - c.min()) / (np.ptp(c) + 1e-8)

fig, axes = plt.subplots(2, 3, figsize=(13, 8.6 * H / W + 1))
axes[0, 0].imshow(gray_f, cmap="gray"); axes[0, 0].set_title("Input (grayscale)")
axes[0, 1].imshow(norm_vis(LL, False), cmap="gray"); axes[0, 1].set_title("LL (approximation)")
axes[0, 2].imshow(norm_vis(LH), cmap="gray"); axes[0, 2].set_title("LH (horizontal detail)")
axes[1, 1].imshow(norm_vis(HL), cmap="gray"); axes[1, 1].set_title("HL (vertical detail)")
axes[1, 2].imshow(norm_vis(HH), cmap="gray"); axes[1, 2].set_title("HH (diagonal detail)")
coeffs_vis = pywt.wavedec2(gray_f, WAVELET, level=LEVEL, mode="periodization")
coeffs_vis = [norm_vis(coeffs_vis[0], False)] + [tuple(norm_vis(b) for b in lv) for lv in coeffs_vis[1:]]
arr, _ = pywt.coeffs_to_array(coeffs_vis)
axes[1, 0].imshow(arr, cmap="gray"); axes[1, 0].set_title(f"{LEVEL}-level DWT pyramid ({WAVELET})")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); save_fig(fig, "fig05_dwt_subbands.png")

energy = {k: float(np.sum(v ** 2)) for k, v in zip(["LL", "LH", "HL", "HH"], [LL, LH, HL, HH])}
tot = sum(energy.values())
print("Energy share per sub-band (level 1):", {k: f"{100 * v / tot:.2f}%" for k, v in energy.items()})

## 3.5 Wavelet based restoration — thresholding of coefficients (denoising)
* **Noise estimate:** σ = median(|HH₁|) / 0.6745 (robust MAD estimator on the finest diagonal band, excluding damaged pixels).
* **BayesShrink threshold** per sub-band: T = σ² / σₓ, where σₓ = √max(mean(c²) − σ², 0).
* **Soft thresholding:** c' = sign(c)·max(|c| − T, 0). Small coefficients (mostly noise) are removed; large ones (real edges and brush strokes) are kept.

In [ ]:
def estimate_sigma(ch, dmask):
    _, (_, _, hh) = pywt.dwt2(ch, WAVELET, mode="periodization")
    mm = cv2.resize(dmask.astype(np.uint8), (hh.shape[1], hh.shape[0]), interpolation=cv2.INTER_NEAREST)
    mm = cv2.dilate(mm, np.ones((5, 5), np.uint8)).astype(bool)
    vals = hh[~mm] if (~mm).sum() > 100 else hh.ravel()
    return float(np.median(np.abs(vals)) / 0.6745)

def threshold_value(c, sigma, rule, n_pixels):
    if rule == "visu":
        return sigma * np.sqrt(2 * np.log(n_pixels))
    sx = np.sqrt(max(np.mean(c ** 2) - sigma ** 2, 1e-12))
    return sigma ** 2 / sx

def wavelet_threshold(coeffs, sigma, n_pixels, mode=THRESH_MODE, rule=THRESH_RULE, gain=DETAIL_GAIN, log=None):
    new = [coeffs[0]]
    for i, lv in enumerate(coeffs[1:]):
        level = len(coeffs) - 1 - i
        bands = []
        for name, c in zip(["LH", "HL", "HH"], lv):
            t = threshold_value(c, sigma, rule, n_pixels)
            c2 = pywt.threshold(c, t, mode) * gain
            if log is not None:
                log.append({"Level": level, "Sub-band": name, "Threshold T": t,
                            "Coefficients zeroed (%)": 100 * np.mean(c2 == 0)})
            bands.append(c2)
        new.append(tuple(bands))
    return new

def dwt_channels(img, wavelet=WAVELET, level=LEVEL):
    return [pywt.wavedec2(img[..., c], wavelet, level=level, mode="periodization") for c in range(3)]

sigmas = [estimate_sigma(damaged[..., c], mask) for c in range(3)]
print("Estimated noise sigma per channel (R, G, B):", np.round(sigmas, 4), "| true sigma:", NOISE_SIGMA)

coeffs = dwt_channels(damaged)                                          # 3.4 on all colour channels
thr_log = []
coeffs_thr = [wavelet_threshold(coeffs[c], sigmas[c], H * W, log=thr_log if c == 1 else None) for c in range(3)]
thr_df = pd.DataFrame(thr_log).round(4)
thr_df.to_csv(os.path.join(OUT_DIR, "table_thresholds_green_channel.csv"), index=False)
display(thr_df)

hh_before = coeffs[1][-1][2].ravel(); hh_after = coeffs_thr[1][-1][2].ravel()
T1 = thr_df.query("Level == 1 and `Sub-band` == 'HH'")["Threshold T"].iloc[0]
fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(-0.3, 0.3, 121)
ax.hist(hh_before, bins=bins, color=GREY, label="Before thresholding")
ax.hist(hh_after[hh_after != 0], bins=bins, color=BLUE, label="After soft thresholding (non-zero)")
ax.axvline(T1, color=INK, lw=1, ls="--"); ax.axvline(-T1, color=INK, lw=1, ls="--")
ax.text(T1, ax.get_ylim()[1] * 0.92, f"  ±T = {T1:.3f}", color=INK)
ax.set_yscale("log"); ax.set_xlabel("HH coefficient value (level 1, green channel)"); ax.set_ylabel("Count (log)")
ax.legend(frameon=False, loc="upper left"); ax.set_title("3.5 Effect of BayesShrink soft thresholding")
plt.tight_layout(); save_fig(fig, "fig06_threshold_histogram.png")

## 3.6 Inverse DWT — reconstruct the image from the modified coefficients

In [ ]:
def idwt_channels(coeffs_list, shape, wavelet=WAVELET):
    out = np.stack([pywt.waverec2(c, wavelet, mode="periodization")[: shape[0], : shape[1]] for c in coeffs_list], -1)
    return np.clip(out, 0, 1).astype(np.float32)

denoised = idwt_channels(coeffs_thr, (H, W))
intact = ~mask & ~gt_mask
noise_before = float(np.std((damaged - original)[intact]))
noise_after  = float(np.std((denoised - original)[intact]))
print(f"Noise level on the intact area (std of error vs original): {noise_before:.4f} -> {noise_after:.4f}"
      f"  ({100 * (1 - noise_after / noise_before):.1f}% reduction)")

y0, x0, sz = densest_window(~mask & ~gt_mask, 160)   # a clean area so the denoising is visible
crop = lambda im: im[y0:y0 + sz, x0:x0 + sz]
show_row([crop(original), crop(damaged), crop(denoised)],
         ["Original (zoom)", "Noisy input (zoom)", "After DWT → threshold → IDWT"], "fig07_wavelet_denoising_zoom.png")

## 3.7 Missing-region reconstruction (inpainting)
Wavelet thresholding removes noise, but it cannot invent content where the painting is physically missing. The damaged regions are therefore filled with **Telea fast-marching inpainting**, which propagates the surrounding (now noise-free) colours and edges into each hole. Inpainting a denoised image avoids smearing noise into the holes.

In [ ]:
def inpaint(img, dmask, radius=INPAINT_RADIUS, method=cv2.INPAINT_TELEA):
    return to_f(cv2.inpaint(to_u8(img), dmask.astype(np.uint8), radius, method))

reconstructed = inpaint(denoised, mask)
show_row([damaged, denoised, reconstructed], ["Damaged input", "After wavelet denoising", "3.7 After missing-region reconstruction"],
         "fig08_reconstruction.png")

## 3.8 Post-processing
* **Seam smoothing (used for evaluation):** an edge-preserving bilateral filter blended only on a thin ring around each filled region, hiding visible boundaries.
* **Display enhancement (optional, reported separately):** CLAHE contrast enhancement + unsharp sharpening. It looks crisper, but it moves pixel values away from the original, so it usually **lowers** PSNR — a useful discussion point.

In [ ]:
def seam_smooth(img, dmask):
    m = dmask.astype(np.uint8); k = np.ones((5, 5), np.uint8)
    ring = (cv2.dilate(m, k) - cv2.erode(m, k)).astype(np.float32)
    alpha = np.clip(cv2.GaussianBlur(ring, (0, 0), 2) * 1.5, 0, 1)[..., None]
    smooth_img = cv2.bilateralFilter(img.astype(np.float32), 7, 0.08, 5)
    return np.clip(alpha * smooth_img + (1 - alpha) * img, 0, 1)

def enhance(img, clip=CLAHE_CLIP, amount=UNSHARP_AMOUNT):
    lab = cv2.cvtColor(to_u8(img), cv2.COLOR_RGB2LAB)
    if clip > 0: lab[..., 0] = cv2.createCLAHE(clip, (8, 8)).apply(lab[..., 0])
    x = to_f(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))
    return np.clip(x + amount * (x - cv2.GaussianBlur(x, (0, 0), 1.0)), 0, 1)

restored = seam_smooth(reconstructed, mask)
restored_enh = enhance(restored)
cv2.imwrite(os.path.join(OUT_DIR, "00_restored.png"), cv2.cvtColor(to_u8(restored), cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(OUT_DIR, "00_restored_enhanced.png"), cv2.cvtColor(to_u8(restored_enh), cv2.COLOR_RGB2BGR))
show_row([restored, restored_enh], ["4. Restored output (seam-smoothed)", "Restored + display enhancement"],
         "fig09_postprocessing.png")

### Pipeline overview figure (for the report / PPT)

In [ ]:
stages = [(gray, "3.1 Pre-processing"), (raw_mask.astype(np.float32), "3.2 Damage detection"),
          (overlay, "3.3 Segmentation"), (arr, "3.4 DWT"), (crop(denoised), "3.5 Thresholding (zoom)"),
          (denoised, "3.6 Inverse DWT"), (reconstructed, "3.7 Reconstruction"), (restored, "3.8 Post-processing")]
fig, axes = plt.subplots(2, 4, figsize=(16, 8.4 * H / W + 1))
for ax, (im, t) in zip(axes.ravel(), stages):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None); ax.set_title(t); ax.axis("off")
plt.tight_layout(); save_fig(fig, "fig10_pipeline_stages.png")

## 5. Evaluation — comparison with baseline methods
All methods use the **same detected mask**. Columns: MSE (0–255 scale, lower is better), PSNR in dB and SSIM (higher is better), plus PSNR measured separately inside the damaged area and on the intact area.

In [ ]:
def wavelet_denoise(img, dmask, wavelet=WAVELET, level=LEVEL, mode=THRESH_MODE, rule=THRESH_RULE, gain=DETAIL_GAIN):
    sig = [estimate_sigma(img[..., c], dmask) for c in range(3)]
    cs = dwt_channels(img, wavelet, level)
    return idwt_channels([wavelet_threshold(cs[c], sig[c], img.shape[0] * img.shape[1], mode, rule, gain) for c in range(3)], img.shape[:2], wavelet)

def proposed(img, dmask, **kw):
    return seam_smooth(inpaint(wavelet_denoise(img, dmask, **kw), dmask), dmask)

def wavelet_iterative_inpaint(img, dmask, iters=40):
    # 'pure wavelet' baseline: holes start from a smooth local average, then are refined by
    # iterative soft-thresholding in the wavelet domain while intact pixels stay fixed (denoised).
    den = wavelet_denoise(img, dmask)
    m = dmask.astype(np.float32)[..., None]
    num = cv2.GaussianBlur(den * (1 - m), (0, 0), 8)
    dnm = cv2.GaussianBlur(1 - dmask.astype(np.float32), (0, 0), 8)[..., None]
    x = np.where(m > 0, num / np.maximum(dnm, 1e-3), den).astype(np.float32)
    for c in range(3):
        ch = x[..., c]
        cs0 = pywt.wavedec2(ch, WAVELET, level=4, mode="periodization")
        lmax = 0.5 * max(np.abs(b).max() for lv in cs0[1:] for b in lv); lmin = 0.5 * sigmas[c]
        for k in range(iters):
            lam = lmax * (lmin / lmax) ** (k / (iters - 1))
            cs = pywt.wavedec2(ch, WAVELET, level=4, mode="periodization")
            cs = [cs[0]] + [tuple(pywt.threshold(b, lam, "soft") for b in lv) for lv in cs[1:]]
            ch[dmask] = pywt.waverec2(cs, WAVELET, mode="periodization")[:H, :W][dmask]
        x[..., c] = ch
    return np.clip(x, 0, 1)

t0 = time.time()
results = {
    "Damaged image (input)":                     damaged,
    "Traditional inpainting - Telea":            inpaint(damaged, mask),
    "Traditional inpainting - Navier-Stokes":    inpaint(damaged, mask, method=cv2.INPAINT_NS),
    "Wavelet denoising only (no inpainting)":    denoised,
    "Wavelet-domain iterative inpainting":       wavelet_iterative_inpaint(damaged, mask),
    "Wavelet + Inpainting (proposed)":           reconstructed,
    "Proposed + seam smoothing (final)":         restored,
    "Proposed + display enhancement":            restored_enh,
}
rows = []
for name, img in results.items():
    mse, psnr, ssim = evaluate(original, img)
    rows.append({"Method": name, "MSE": mse, "PSNR (dB)": psnr, "SSIM": ssim,
                 "PSNR damaged area": evaluate(original, img, gt_mask),
                 "PSNR intact area": evaluate(original, img, ~gt_mask)})
metrics_df = pd.DataFrame(rows).round({"MSE": 2, "PSNR (dB)": 2, "SSIM": 3, "PSNR damaged area": 2, "PSNR intact area": 2})
metrics_df.to_csv(os.path.join(OUT_DIR, "table_method_comparison.csv"), index=False)
print(f"(all methods computed in {time.time() - t0:.1f}s)")
display(metrics_df)

In [ ]:
# visual comparison
show_list = ["Damaged image (input)", "Traditional inpainting - Telea", "Wavelet denoising only (no inpainting)",
             "Wavelet-domain iterative inpainting", "Proposed + seam smoothing (final)"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9.6 * H / W + 1))
axes[0, 0].imshow(original); axes[0, 0].set_title("Original (reference)")
for ax, k in zip(axes.ravel()[1:], show_list):
    _, p, s = evaluate(original, results[k]); ax.imshow(results[k]); ax.set_title(f"{k}\nPSNR {p:.2f} dB | SSIM {s:.3f}", fontsize=9.5)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); save_fig(fig, "fig11_method_comparison.png")

# zoomed crops on the most damaged area
yz, xz, sz = densest_window(gt_mask, 180)
cz = lambda im: im[yz:yz + sz, xz:xz + sz]
show_row([cz(original), cz(damaged), cz(results["Traditional inpainting - Telea"]), cz(restored)],
         ["Original", "Damaged", "Telea inpainting", "Proposed (final)"], "fig12_zoom_most_damaged.png", height=3.6)

# absolute error maps
errs = [np.abs(original - results[k]).mean(-1) for k in ["Damaged image (input)", "Traditional inpainting - Telea", "Proposed + seam smoothing (final)"]]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6 * H / W + 0.6))
for ax, e, t in zip(axes, errs, ["|error| damaged", "|error| Telea", "|error| proposed"]):
    im = ax.imshow(e, cmap="magma", vmin=0, vmax=0.3); ax.set_title(t); ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01, label="mean absolute error (0-1)")
save_fig(fig, "fig13_error_maps.png")

In [ ]:
# metric bar charts - one chart per metric (no dual axes), proposed highlighted
plot_methods = ["Damaged image (input)", "Traditional inpainting - Telea", "Traditional inpainting - Navier-Stokes",
                "Wavelet denoising only (no inpainting)", "Wavelet-domain iterative inpainting", "Proposed + seam smoothing (final)"]
short = ["Damaged\ninput", "Telea", "Navier-\nStokes", "Wavelet\nonly", "Wavelet\niterative", "Proposed"]
sub = metrics_df.set_index("Method").loc[plot_methods]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col, fmt, note in zip(axes, ["PSNR (dB)", "SSIM", "MSE"], ["{:.2f}", "{:.3f}", "{:.1f}"],
                              ["higher is better", "higher is better", "lower is better"]):
    vals = sub[col].values
    colors = [BLUE if "Proposed" in m else GREY for m in plot_methods]
    bars = ax.bar(short, vals, color=colors, width=0.62, edgecolor="white", linewidth=2)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, fmt.format(v), ha="center", va="bottom", fontsize=9, color=INK)
    ax.set_title(f"{col} ({note})"); ax.grid(axis="y"); ax.set_axisbelow(True)
    ax.tick_params(axis="x", labelsize=8.5)
plt.tight_layout(); save_fig(fig, "fig14_metrics_bars.png")

## 6. Experiments (evidence for the analysis section)
**E1** wavelet family × decomposition level · **E2** thresholding rule · **E3** noise level · **E4** amount of damage. Each experiment changes one factor and keeps everything else fixed.

In [ ]:
# E1: wavelet family x level
e1 = []
for w in ["haar", "db2", "db4", "sym4", "coif2", "bior2.2"]:
    for L in [1, 2, 3, 4]:
        _, p, s = evaluate(original, proposed(damaged, mask, wavelet=w, level=L))
        e1.append({"Wavelet": w, "Level": L, "PSNR (dB)": round(p, 2), "SSIM": round(s, 3)})
e1_df = pd.DataFrame(e1); e1_df.to_csv(os.path.join(OUT_DIR, "table_E1_wavelet_family_level.csv"), index=False)
display(e1_df.pivot(index="Wavelet", columns="Level", values="PSNR (dB)"))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
lv3 = e1_df[e1_df.Level == LEVEL]
best = lv3["PSNR (dB)"].idxmax()
axes[0].bar(lv3.Wavelet, lv3["PSNR (dB)"], color=[BLUE if i == best else GREY for i in lv3.index], width=0.6, edgecolor="white", linewidth=2)
for x, v in zip(lv3.Wavelet, lv3["PSNR (dB)"]): axes[0].text(x, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(lv3["PSNR (dB)"].min() - 1, lv3["PSNR (dB)"].max() + 0.5)
axes[0].set_title(f"E1a: PSNR by wavelet family (level {LEVEL})"); axes[0].set_ylabel("PSNR (dB)"); axes[0].grid(axis="y"); axes[0].set_axisbelow(True)
dbw = e1_df[e1_df.Wavelet == WAVELET]
axes[1].plot(dbw.Level, dbw["PSNR (dB)"], color=BLUE, lw=2, marker="o", ms=7)
for x, v in zip(dbw.Level, dbw["PSNR (dB)"]): axes[1].text(x, v + 0.03, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
axes[1].set_ylim(dbw["PSNR (dB)"].min() - 0.08, dbw["PSNR (dB)"].max() + 0.12)
axes[1].set_xticks(dbw.Level); axes[1].set_xlabel("Decomposition level"); axes[1].set_ylabel("PSNR (dB)")
axes[1].set_title(f"E1b: PSNR vs decomposition level ({WAVELET})"); axes[1].grid(axis="y"); axes[1].set_axisbelow(True)
plt.tight_layout(); save_fig(fig, "fig15_E1_wavelet_sweep.png")

# E2: thresholding rule and detail gain
e2 = []
for mode in ["soft", "hard"]:
    for rule in ["bayes", "visu"]:
        _, p, s = evaluate(original, proposed(damaged, mask, mode=mode, rule=rule))
        e2.append({"Setting": f"{mode} / {'BayesShrink' if rule == 'bayes' else 'VisuShrink'}", "PSNR (dB)": round(p, 2), "SSIM": round(s, 3)})
for g in [1.1, 1.2]:
    _, p, s = evaluate(original, proposed(damaged, mask, gain=g))
    e2.append({"Setting": f"soft / BayesShrink + detail gain {g}", "PSNR (dB)": round(p, 2), "SSIM": round(s, 3)})
e2_df = pd.DataFrame(e2); e2_df.to_csv(os.path.join(OUT_DIR, "table_E2_threshold_rules.csv"), index=False)
display(e2_df)

In [ ]:
# E3 (noise level) and E4 (damage coverage): re-simulate, re-detect, re-restore
def run_case(noise, coverage, seed=SEED):
    global rng
    rng = np.random.default_rng(seed)
    dmg, gt = simulate_damage(original, coverage=coverage, noise_sigma=noise)
    _, sm, md_ = preprocess(dmg); r_, l_, _ = detect_damage(sm, md_)
    msk = segment(r_ | l_)[0]
    out = {}
    for name, img in [("Damaged input", dmg), ("Telea inpainting", inpaint(dmg, msk)), ("Proposed", proposed(dmg, msk))]:
        _, p, s = evaluate(original, img); out[name] = (p, s)
    return out

e3 = []
for nz in [0.02, 0.04, 0.06, 0.08, 0.10]:
    for k, (p, s) in run_case(nz, HOLE_COVERAGE).items():
        e3.append({"Noise sigma": nz, "Method": k, "PSNR (dB)": round(p, 2), "SSIM": round(s, 3)})
e4 = []
for cv in [0.05, 0.10, 0.15, 0.20, 0.25]:
    for k, (p, s) in run_case(NOISE_SIGMA, cv).items():
        e4.append({"Hole coverage": cv, "Method": k, "PSNR (dB)": round(p, 2), "SSIM": round(s, 3)})
e3_df, e4_df = pd.DataFrame(e3), pd.DataFrame(e4)
e3_df.to_csv(os.path.join(OUT_DIR, "table_E3_noise_sweep.csv"), index=False)
e4_df.to_csv(os.path.join(OUT_DIR, "table_E4_damage_sweep.csv"), index=False)
display(e3_df.pivot(index="Noise sigma", columns="Method", values="PSNR (dB)"))
display(e4_df.pivot(index="Hole coverage", columns="Method", values="PSNR (dB)"))

series = [("Proposed", BLUE), ("Telea inpainting", ORANGE), ("Damaged input", AQUA)]
fig, axes = plt.subplots(2, 2, figsize=(13, 7.6))
for row, (df, xcol, xlabel) in enumerate([(e3_df, "Noise sigma", "Noise sigma (0-1 scale)"),
                                          (e4_df, "Hole coverage", "Hole coverage (fraction of image)")]):
    for col, metric in enumerate(["PSNR (dB)", "SSIM"]):
        ax = axes[row, col]
        for name, color in series:
            d = df[df.Method == name]
            ax.plot(d[xcol], d[metric], color=color, lw=2, marker="o", ms=7, label=name)
            ax.text(d[xcol].iloc[-1], d[metric].iloc[-1], "  " + name, color=INK, fontsize=8.5, va="center")
        ax.set_xlabel(xlabel); ax.set_ylabel(metric); ax.grid(axis="y"); ax.set_axisbelow(True)
        ax.set_title(f"{'E3' if row == 0 else 'E4'}: {metric} vs {xcol.lower()}")
        ax.set_xticks(sorted(df[xcol].unique())); ax.set_xlim(df[xcol].min() - 0.005, df[xcol].max() * 1.32)
handles, labels_ = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.03))
plt.tight_layout(); save_fig(fig, "fig16_E3_E4_sweeps.png")
rng = np.random.default_rng(SEED)

## 7. Final result figure and report-ready summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.6 * H / W + 0.8))
for ax, im, t in zip(axes, [original, damaged, restored], ["1. Original painting", "2. Damaged painting (input)", "4. Digitally restored painting"]):
    ax.imshow(im); ax.set_title(t, fontsize=12); ax.axis("off")
plt.tight_layout(); save_fig(fig, "fig17_final_triptych.png")

r = metrics_df.set_index("Method")
d, t, p = r.loc["Damaged image (input)"], r.loc["Traditional inpainting - Telea"], r.loc["Proposed + seam smoothing (final)"]
best_e1 = e1_df.loc[e1_df["PSNR (dB)"].idxmax()]
summary = f'''
REPORT-READY SUMMARY  ({IMAGE_NAME}, {W}x{H}, seed {SEED})
-------------------------------------------------------------------
Damage: {gt_mask.mean()*100:.1f}% of the painting missing/torn/scratched + Gaussian noise sigma = {NOISE_SIGMA}
Detection: {n_regions} damaged regions | precision {precision:.3f} | recall {recall:.3f} | IoU {iou:.3f} | Dice {dice:.3f}
           false alarm on the undamaged original: {false_alarm*100:.2f}% of pixels
Noise: true sigma {NOISE_SIGMA:.3f}, MAD estimate {np.mean(sigmas):.4f}; error std on intact area {noise_before:.4f} -> {noise_after:.4f} after wavelet thresholding

                         MSE      PSNR(dB)   SSIM
Damaged input        {d["MSE"]:8.2f}   {d["PSNR (dB)"]:7.2f}   {d["SSIM"]:.3f}
Traditional (Telea)  {t["MSE"]:8.2f}   {t["PSNR (dB)"]:7.2f}   {t["SSIM"]:.3f}
Wavelet + Inpainting {p["MSE"]:8.2f}   {p["PSNR (dB)"]:7.2f}   {p["SSIM"]:.3f}

Gain over damaged input : +{p["PSNR (dB)"]-d["PSNR (dB)"]:.2f} dB PSNR, SSIM +{p["SSIM"]-d["SSIM"]:.3f}, MSE -{100*(1-p["MSE"]/d["MSE"]):.1f}%
Gain over Telea alone   : +{p["PSNR (dB)"]-t["PSNR (dB)"]:.2f} dB PSNR, SSIM +{p["SSIM"]-t["SSIM"]:.3f}, MSE -{100*(1-p["MSE"]/t["MSE"]):.1f}%
Inside the damaged area : Telea {t["PSNR damaged area"]:.2f} dB vs proposed {p["PSNR damaged area"]:.2f} dB
On the intact area      : Telea {t["PSNR intact area"]:.2f} dB vs proposed {p["PSNR intact area"]:.2f} dB
Best wavelet setting (E1): {best_e1.Wavelet}, level {best_e1.Level} -> {best_e1["PSNR (dB)"]:.2f} dB
'''
print(summary)
with open(os.path.join(OUT_DIR, "summary.txt"), "w") as fh:
    fh.write(summary)

## 8. Download all results (figures + CSV tables) as one zip

In [ ]:
zip_path = os.path.join(os.path.dirname(OUT_DIR) or ".", "dip_wavelet_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, f), arcname=f)
print(f"{len(os.listdir(OUT_DIR))} files ->", zip_path)
for f in sorted(os.listdir(OUT_DIR)): print("  ", f)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 9. (Optional) Restore your own already-damaged photo — no ground truth
Set `RUN_REAL_PHOTO = True` and run this cell. Upload a photo of a damaged painting. Without an undamaged reference, PSNR/SSIM cannot be computed, so the cell reports the damaged area and the noise estimate before and after instead. If missing regions in your photo are not near-white, adjust `V_THR` / `S_THR` in the Configuration cell.

In [ ]:
RUN_REAL_PHOTO = False

if RUN_REAL_PHOTO and IN_COLAB:
    from google.colab import files
    up = files.upload(); name = next(iter(up))
    real = to_f(prepare(decode(up[name])))
    _, sm, md_ = preprocess(real); r_, l_, _ = detect_damage(sm, md_)
    real_mask, n_real, _, _ = segment(r_ | l_)
    real_restored = proposed(real, real_mask)
    from skimage.restoration import estimate_sigma as sk_sigma
    s_before = sk_sigma(real, channel_axis=-1, average_sigmas=True)
    s_after  = sk_sigma(real_restored, channel_axis=-1, average_sigmas=True)
    ov = real.copy(); ov[real_mask] = 0.6 * ov[real_mask] + 0.4 * np.array([0.89, 0.12, 0.12])
    print(f"Detected damage: {real_mask.mean()*100:.2f}% in {n_real} regions | noise estimate {s_before:.4f} -> {s_after:.4f}")
    show_row([real, ov, real_restored], ["Your damaged photo", "Detected damage (red)", "Restored"], "fig18_real_photo.png")
else:
    print("Skipped (set RUN_REAL_PHOTO = True and run in Colab).")